## Introduzione

L'Information Retrieval (IR) è il campo dell'informatica che studia come trovare informazioni rilevanti all'interno di grandi documenti non strutturati (soppratutto testo). 

Alcuni esempi di applicazioni di IR includono:
- Motori di ricerca (Google, Bing, etc.)
- Ricerca nelle email
- Ricerca nei file del computer
- Archivi aziendali
- Database giuridici

L'obiettivo principale è **recuperare i documenti più rilevanti rispetto al bisogno informativo dell'utente**.

L'importanza dell'IR è cresciuta esponenzialmente con l'aumento della quantità di informazioni disponibili online: attualmente circa 1.3 miliardi di siti web di cui solo 15% attivi e oltre un milione di nuovi url creati ogni giorno. Ciò rende fondamentale avere sistemi efficienti per filtrare e trovare le informazioni desiderate.

Importante conoscere la differenza tra dati strutturati e non strutturati:
- **Dati strutturati**: organizzati in un formato predefinito (es. database relazionali, fogli di calcolo)
- **Dati non strutturati**: non organizzati in un formato predefinito (es.pagine web, email, documenti di testo)

Un sistema di IR considera: 
- **Collection**: insieme di documenti da cui recuperare informazioni
- **Query**: richiesta dell'utente che esprime il bisogno informativo
- **Goal**: trovare i documenti più rilevanti rispetto alla query

<img src="img/diff.png" width="400px" alt="differenza tra dati strutturati e non strutturati">

Uno dei modelli base di IR è **Bag of Words**. L'idea è che ogni documento rappresenta null'altro che un insieme di parole, senza considerare l'ordine o la grammatica. In questo senso alcune delle tecniche usate sono **stop words** (parole inutili e non significanti vengono ignorate) e **stemming** (riduzione delle parole alla loro radice).

Per capire quanto un documento è rilevante rispetto a una query, si usano diverse statistiche come la frequenza della parola in un document (TF - Term Frequency) e la frequenza inversa della parola in tutti i documenti (IDF - Inverse Document Frequency). IDF rappresenta la rarità di una parola nell'intera collezione di documenti, più è rara più è importante per identificare un documento specifico.

### **Lez 1**
**IR** == trovare **materiale** (usually docs) di natura **non strutturata** (usually text), **a partire da una grande collection**, che soddisfino il **bisogno informativo** dell'utente

- **Collection**: insieme di documenti (per ora assumiamo che siano statici)
- **Goal**: trovare i documenti che abbiano informazioni **rilevanti** per il bisogno informativo dell'utente, e che quindi lo aiutino a completare un **task**

Il classico modello adottato in un motore di ricerca è il seguente:

<img src="img/csmodel.png" width="500">

Per cui al fine di risolvere un task, lo user ha un information need e per soddisfarlo scrive una **query** nel motore di ricerca. Si noti come tra user task ed info need può esservi una misconception (ossia, lo user potrebbe non essere in grado di esprimere correttamente il proprio bisogno informativo) e quindi una misformulation tra info need e query. Una volta fatta la query, il search engine estrae (qui l'IR) i documenti più rilevanti e quindi mostra i risultati all'utente, che se non soddisfatto può fare query refinement.

Un concetto fondamentale che approfondiremo è quello di **misurare le performance** per capire se un sistema di IR è buono o meno. Le due metriche più importanti in questo senso sono:
- **Precision**: percentuale di documenti recuperati che sono rilevanti (that is, tra i documenti che il sistema ha recuperato (TP + FP), quanti sono effettivamente rilevanti (TP))
- **Recall**: percentuale di documenti rilevanti che sono stati recuperati (that is, tra i documenti che sono effettivamente rilevanti (TP + FN), quanti sono stati recuperati (TP))

### Term-Document incidence matrix
Vogliamo estrarre dalle opere di Shakespeare i documenti che contengono le parole Caesar AND Brutus, ma NOT Calpurnia. L'idea Naive sarebbe quella di fare una scansione lineare di tutti i documenti (grep) e verificare se contengono o meno le parole. Tuttavia, questo approccio è inefficiente, soprattutto quando la collezione di documenti è molto grande.

Un modo migliore per rappresentare la relazione tra termini e documenti è attraverso una **matrice di incidenza termine-documento**. In questa matrice, le righe rappresentano i termini (parole) e le colonne rappresentano i documenti. Ogni cella contiene un valore binario (0 o 1) che indica se il termine è presente o meno nel documento.

Immaginando di voler recuperare documenti che contengono le parole Caesar AND Brutus, ma NOT Calpurnia, l'idea potrebbe essere di **estrarre** le righe corrispondenti a Caesar, Brutus e Calpurnia, e poi **combinare** i risultati usando operazioni logiche (nel nostro caso AND e NOT).

<img src="img/tdmatrix.png" width="400">

Tuttavia c'è un problema: se la collezione di documenti è molto grande, la matrice di incidenza può diventare estremamente grande e sparsa (la maggior parte delle celle saranno 0). Questo rende l'approccio inefficiente sia in termini di spazio che di tempo. 

ex. N = 1 milione di documenti, dove ogni documento ha circa 1000 parole. Immaginiamo che in media ogni parola richieda 6 byte -> 6 * 1000 * 1 milione = 6 GB di dati nei documenti. Ipotizzando ora un totale di 500k parole distinte, la matrice di incidenza avrebbe 500k righe e 1 milione di colonne, con un totale di 500 miliardi di celle (that is, half a trillion). Tuttavia questa matrice non avrebbe più di un miliardo di 1 (considerando che ogni documento ha circa 1000 parole), il che significa che sarebbe estremamente sparsa (99.8% di celle con valore 0).

## **Inverted Index**

Quindi un modo decisamente migliore per rappresentare questa matrice è quello di usare una **lista di parole** (**inverted index**), dove per ogni parola si tiene traccia dei documenti in cui essa appare. In questo modo, invece di dover memorizzare una matrice enorme e sparsa, possiamo memorizzare solo le parole che effettivamente appaiono nei documenti e i documenti in cui appaiono (solo gli 1 della matrice di incidenza).

NB. densa se rappresentazione matriciale, sparsa se rappresentazione a lista di parole (inverted index).

**Inverted Index** == for each term t, we store a list of all documents that contain t. Each doc is identified by a unique docID, and the list of docIDs is called the **posting list** for that term (and each document is a posting).

Dal momento che una parola potrebbe apparire in molti documenti mentre un'altra in pochissimi, è necessario avere delle posting list di lunghezza variabile:
- su disco, le posting list sono memorizzate in modo contiguo (per permettere letture sequenziali efficienti), esattamente come si vede nell'immagine (dal momento che in disco restano statiche).
- in memoria RAM invece si possono usare strutture dati più flessibili (es. linked list, array dinamici) per gestire le posting list di lunghezza variabile.

Per gestire il tutto, si sfrutta un **Dizionario** con chiavi i termini e valori le posting list. Se volessi cercare la posting list di una parola, dovrei anzitutto cercare la parola nel dizionario (O(log n) se i termini sono ordinati) per poi accedere alla posting list associata (O(1)).

Importante puntualizzare già da adesso come anche i DocID siano **ordinati** in ordine non decrescente all'interno di ogni posting list, in modo da permettere operazioni efficienti di merging (AND, OR, NOT) tra posting list diverse. Nell'immagine ad esempio se volessi trovare tutti i documenti con Brutus AND Caesar, prima vedo 1 comune ok. Poi 2 comune ok. Poi 4 ok. Poi 11 vs 5: fisso l'11 e scorro il 5 fino a trovare un match o superare l'11. Non trovo match in quanto arrivo a 16 e fisso il 16, quindi scorro sopra etc...

<img src="img/invertedindex.png" width="400">

#### Inverted Index: costruzione

<img src="img/indexing.png" width="400">

Si parte da una collezione di documenti e si usa una serie di tecniche, che però devono essere utilizzate con cautela e in base al contesto. 
1. **Tokenizer**: il primo passo è quello di suddividere i documenti in unità più piccole chiamate token (solitamente parole). Ad esempio, la frase "Il gatto è sul tavolo" verrebbe tokenizzata in ["Il", "gatto", "è", "sul", "tavolo"]. NB. un esempio di uso "errato" di tokenizzazione è per parole composte come "New York" o "lenti a contatto".
2. **Linguistic Modules**: dopo la tokenizzazione, si possono applicare diverse tecniche linguistiche per migliorare la qualità dell'indice. Ad esempio:
   - **Normalization**: conversione di tutte le parole per evitare duplicati (es. "Gatto" e "gatto" diventano "gatto"). Esempio di uso "errato" di normalizzazione è per parole che hanno significati diversi a seconda del caso (es. USA come paese si "confonde" con usa verbo).
   - **Stop Words**: parole comuni che non aggiungono significato (es. "il", "e", "ma") vengono rimosse.
   - **Stemming**: riduzione delle parole alla loro radice (es. authorize, authorization, authorized diventano "author"). NB qui possono esserci problemi es. Un giorno feci una partita, feci si confonde tra fare e feci.
3. Dopo queste trasformazioni otteniamo i **Modified Tokens**. Solo a questo punto entra in gioco l'**Indexer**, che costruisce **l'inverted index** a partire dai modified tokens. 



Una volta ottenuti i modified tokens, l'indexer costruisce l'inverted index associando ogni token ai documenti in cui appare. Anzitutto si creano le coppie (term, docID) per ogni token e documento in cui appare. Successivamente, si ordinano queste coppie per termine e docID, in modo da poter costruire efficientemente le posting list. Dopo il sort si ottiene quindi qualcosa del tipo:
```
(brutus,1)
(brutus,2)
(caesar,1)
(caesar,2)
(caesar,2)
(enact,1)
(julius,1)
```
Dopodiché i dati sono separati in due strutture:
- Dizionario, che contiene tutti i termini distinti (es. brutus, caesar, enact, julius, etc.). **Inoltre viene anche salvata la document frequency, ossia per ogni termine il numero di documenti in cui appare**. La frequenza sarà molto importante per fare ranking dei risultati (termini più rari sono più importanti per identificare un documento specifico), ottimizzare le query (es. se una query contiene un termine molto raro, è più efficiente partire da quello) e calcolare TF-IDF.
- Posting list, che contiene per ogni termine la lista dei documenti in cui appare (es. per brutus -> [1, 2], per caesar -> [1, 2, 2], etc.)

Si noti che se una parola appare più volte in un documento, questa viene registrata una sola volta nella posting list. 

<img src="img/dictionary.png" width="400">

In questo senso, in memoria si spende principalmente per mantenere:
- i termini distinti nel dizionario
- le posting list per ogni termine, che contengono i docID dei documenti in cui il termine appare
- i puntatori alle posting list, che permettono di accedere efficientemente alle posting list a partire dal dizionario


### Inverted Index: query processing (Lez 3)
Ora ci chiediamo come processare una query usando l'inverted index.
Supponiamo di avere una query del tipo "Brutus AND Caesar". Come già anticipato per processare una query di questo tipo dobbiamo anzitutto accedere alle posting list di Brutus e Caesar, e poi fare un **merge** tra le due posting list per trovare i documenti che contengono entrambi i termini (si scorrono le due posting list in parallelo e si avanza in quella che ha il docID più piccolo fino a trovare un match o superare l'altro).

es. se la posting list di Brutus è [1, 2, 4, 11, 16] e quella di Caesar è [1, 2, 5, 6, 11], allora:
- confronto 1 vs 1 -> match, aggiungo 1 al risultato
- confronto 2 vs 2 -> match, aggiungo 2 al risultato
- confronto 4 vs 5 -> non match, avanzo in Brutus (4 < 5)
- confronto 11 vs 5 -> non match, avanzo in Caesar (5 < 11)
- confronto 11 vs 6 -> non match, avanzo in Caesar (6 < 11)
- confronto 11 vs 11 -> match, aggiungo 11 al risultato

Ipotizzando che la prima lista abbia x elementi e la seconda y, il tempo di esecuzione del merge è **O(x + y)** (in quanto si scorre ogni lista al massimo una volta).
**Ovviamente per far si che il merge sia possibile è CRUCIALE che i docID siano ordinati all'interno delle posting list.**

```python
INTERSECT(p1, p2):
answer ← {}

while p1 ≠ NIL and p2 ≠ NIL

    if docID(p1) = docID(p2)
        add docID to answer
        p1 ← next(p1)
        p2 ← next(p2)

    else if docID(p1) < docID(p2)
        p1 ← next(p1)

    else
        p2 ← next(p2)

return answer
```

#### **Boolean Retrieval Model**
Si passa quindi ad uno dei modelli più semplici di IR, ossia il **Boolean Retrieval Model**. In questo modello, una query è espressa come una formula booleana composta da termini e operatori logici (AND, OR, NOT). Il sistema restituisce i documenti che soddisfano la formula booleana.
Ad esempio "Brutus OR NOT Calpurnia" restituirebbe tutti i documenti che contengono Brutus o che non contengono Calpurnia.

Questo modello ha due caratteristiche principali:
- Considera i documenti semplicemente come insieme di parole, senza considerare l'ordine o la grammatica (bag of words)
- Il sistema non considera quanto spesso una parola compare, né la sua posizione all'interno del documento. L'unica cosa che conta è se la parola è presente o meno nel documento. Si parla in questo senso di **Exact Match**: un documento viene restituito se e solo se soddisfa esattamente la formula booleana della query.

Questo approccio è stato per anni il modello dominante nei sistemi di IR, ma ha dei limiti evidenti. Infatti è molto poco flessibile e non tiene conto della rilevanza dei documenti rispetto alla query, non permettendo quindi di fare ranking (tutti i documenti che soddisfano la formula booleana vengono restituiti, indipendentemente da quanto siano rilevanti). **Boolean è quindi assolutamente inadatto per motori di ricerca generici come Google.**

Tuttavia ncora oggi molti sistemi di IR usano ancora Boolean, come ad esempio macOS Spotlight, Email search, library catalogues. Infatti l'exact match permette di avere risultati molto precisi ed **è utile quando l'utente sa esattamente cosa sta cercando.**

**NB**. Ricordando che la Precision è il numero di documenti rilevanti su tutti i documenti restituiti mentre la Recall è il numero di documenti rilevanti restituiti su tutti i documenti rilevanti, il Boolean Retrieval Model tende ad avere una Precision molto alta (in quanto restituisce solo documenti che soddisfano esattamente la formula booleana, quindi mi aspetto che molti dei documenti restituiti siano rilevanti) ma una Recall più bassa (in quanto alcuni documenti rilevanti potrebbero non soddisfare esattamente la formula booleana e quindi non essere restituiti).

Un famoso esempio di applicazione del Boolean Retrieval Model è Westlaw: si tratta di un database giuridico che contiene milioni di documenti legali (sentenze, leggi, etc.) e che permette agli avvocati di cercare documenti rilevanti per i loro casi. Westlaw utilizza un sistema di IR basato su Boolean Retrieval Model, in quanto gli avvocati spesso sanno esattamente cosa stanno cercando (es. una sentenza specifica) e quindi l'exact match è particolarmente utile in questo contesto. 

#### Implementing Boolean Queries and Optimization
Vediamo ora come adattare il merge per gli altri operatori logici (OR, NOT). Per l'OR, l'idea è di restituire tutti i documenti che appaiono in almeno una delle due posting list. Quindi si scorre in parallelo le due posting list e si aggiunge al risultato ogni documento che appare in almeno una delle due liste. Il tempo di esecuzione è sempre **O(x + y)**.

```python
UNION(p1, p2):
answer ← {}
while p1 ≠ NIL and p2 ≠ NIL

    if docID(p1) = docID(p2)
        add docID to answer
        p1 ← next(p1)
        p2 ← next(p2)

    else if docID(p1) < docID(p2)
        add docID(p1) to answer
        p1 ← next(p1)

    else
        add docID(p2) to answer
        p2 ← next(p2)
```

Per il NOT dobbiamo considerare tre casistiche:
- **NOT A**: restituisce tutti i documenti che non contengono A. In questo caso si scorre la posting list di A e si aggiungono al risultato tutti i documenti che non appaiono in quella lista. Bisogna quindi conoscere tutta la collezione di documenti per poter restituire quelli che non contengono A -> **O(N)**, dove N è il numero totale di documenti nella collezione.
- **A AND NOT B**: restituisce tutti i documenti che contengono A ma non B. In questo caso si scorre la posting list di A e si aggiungono al risultato solo i documenti che non appaiono nella posting list di B -> è sufficiente riadattare il merge dove 
    - scorriamo entrambe le liste
    - se troviamo un elemento in comune lo scartiamo
    - quando un docID è solo in A lo aggiungiamo al risultato
Il tempo di esecuzione è **O(x + y)**, dove x è la lunghezza della posting list di A e y è la lunghezza della posting list di B.
- **A OR NOT B**: restituisce tutti i documenti che contengono A o non contengono B. Qui ci servono tutti i documenti che non contengono B, quindi è necessario conoscere tutta la collezione di documenti. Una volta ottenuti i documenti che non contengono B, dobbiamo fare l'OR con la posting list di A, per un tempo totale di **O(N)**.

Man mano che le query si fanno più complesse, è necessario fare più merge tra posting list diverse (es. (Brutus OR Caesar) AND NOT (Antony OR Calpurnia)). In questi casi è possibile **ottimizzare l'esecuzione della query** scegliendo l'ordine in cui fare i merge. Vediamo un esempio:

Brutus AND Calpurnia AND Caesar, dove 
Brutus:      1 2 3 5 8 16 21 34
Caesar:      2 4 8 16 32 64 128
Calpurnia:   13 16.

Si osserva che Calpurnia è il termine più raro (appare in meno documenti), e dato che quando si effettua il merge tra A e B il risultato non può essere più grande della posting list più corta, è più efficiente partire da Calpurnia e fare prima il merge con Brutus, per poi fare il merge con Caesar. In questo modo si riduce il numero di documenti da considerare nei merge successivi, ottimizzando l'esecuzione della query.

Quindi nel nostro caso Calpurnia AND Brutus AND Caesar si esegue come (Calpurnia AND Brutus) AND Caesar, e in generale **per eseguire query con molti AND bisogna intersecare prima le posting lists dei termini più rari**.
Come capisco che un termine è più raro di un altro? Semplice, **guardando la document frequency (DF) che è salvata nel dizionario per ogni termine** (per questo prima l'abbiamo salvata!).

**E se avessi una query del tipo (A OR B) AND (C OR D) AND (E OR F)?** In questo caso si procede ugualmente a prima, solo che si deve stimare il numero di documenti per ogni OR: usando un approccio conservativo, si può stimare che il numero di documenti restituiti da un OR sia al massimo la somma delle lunghezze delle posting list dei termini coinvolti (es. dim(A OR B) = df(A) + df(B)).

<img src="img/optimization.png" width="400">

Qui eyes + kaleidoscope = 300321, marmalade + skies = 379571 e tangerine + trees = 363465. Quindi l'ordine di esecuzione più efficiente è:
(eyes OR kaleidoscope) AND (tangerine OR trees) AND (marmalade OR skies).

E se invece avessi una query del tipo **friends AND romans AND (NOT countrymen)**? Come possiamo usare la frequenza di countrymen?
In questo caso bisogna ragionare "al contrario": se countrymen è un termine molto raro, allora romans AND NOT countrymen restituirà molti documenti, quasi pari alla dimensione della posting list di romans. Al contrario, se countrymen è un termine molto comune, allora romans AND NOT countrymen restituirà pochi documenti, in quanto la maggior parte dei documenti che contengono romans conterranno anche countrymen. Quindi se countrymen è termine raro conviene eseguire prima friends AND romans per ridurre il numero di candidati, mentre se countrymen è termine comune conviene eseguire prima romans AND NOT countrymen. (Ricorda che l'ottimizzazione delle query boolean è basata sulla regola di fare prima le operazioni che restituiscono meno documenti, in modo da ridurre il numero di documenti da considerare nei merge successivi).

Exercise: Extend the merge to an arbitrary Boolean
query. Can we always guarantee execution in time
linear in the total postings size?

No, non sempre possiamo garantire tempo lineare nella somma delle posting lists dei termini della query in quanto, come detto prima, potremmo avere qualcosa del tipo A OR NOT B, che richiede di considerare tutta la collezione di documenti (O(N))

### **Phrase Queries and Positional Indexes**
Cosa succede se però volessi cercare non più una parola, ma una frase esatta? Per esempio, "standford University". Con il metodo attuale restituirei anche frasi del tipo "I went to university at standford", per via dell'exact match. 

Il problema è che con l'inverted index classico abbiamo solo term -> list of docs, quindi sappiamo in quali documenti una parola appare ma non dove appare. Una prima soluzione naive proposta è quella di usare un **biword index**, ossia si aggiunge all'indice (oltre ai single terms) ogni coppia di parole consecutive (biwords). Per esempio per il testo Friends, Romans, Countrymen avermmo i biword friends romans e romans countrymen. 

Con il biword index l'indice non diventa eccessivamente lungo, dal momento che il numero di coppie consecutive in un insieme di cardinalità n è al massimo n-1 (lineare in n). Tuttavia, il biword index ha un limite evidente: se volessi cercare una frase più lunga, come ad esempio "standford university palo alto" non mi basterebbero più biwords.

Un'idea potrebbe essere mantenere biword e fare la query "standford university palo alto" come "standford university" AND "university palo" AND "palo alto". Tuttavia questa query può restituire falsi positivi dal momento che un documento potrebbe contenere le tre biwords ma non la frase esatta (es. "I went to standford university and years later my sister went to university palo alto").

Per evitare falsi positivi dovrei indicizzare n-grammi più grandi, ma ciò porterebbe a un indice molto grande e poco efficiente.

La soluzione al problema, usata nei motori di ricerca moderni, è il **positional index**. L'idea è di mantenere per ogni termine non solo la lista dei documenti in cui appare, ma anche le posizioni all'interno di ogni documento in cui il termine appare. In questo modo, quando voglio cercare una frase esatta, posso verificare se i termini della frase appaiono nelle posizioni corrette all'interno dei documenti.

Vediamo un esempio per capire come si verifica una frase, "to be or not to be", usando un positional index. L'idea è semplicemente di prendere le posting list di tutti i termini della frase (to, be, or, not) e poi verificare se nei documenti in cui appaiono i termini, questi appaiono in posizioni consecutive.

<img src="img/positionalindex.png" width="400">

Nell'immagine sopra si vede come il doc 4 possiede un "to be" dal momento che in posizione 15 ha to e 17 be. Come verificare nel pratico? **Si utilizza di nuovo il merge**: anzitutto ovviamente si usa il merge per trovare i documenti che contengono tutti i termini della frase (to, be, or, not). Successivamente si fa il **merge sulle posizioni**: la procedura è identica solo che si aggiunge il documento tra i candidati solo se i termini appaiono in posizioni consecutive.

es. to : 8,16,190,429,433 be : 17,191,291,430,434
17 = 8+1? no, quindi avanzo il puntatore di to (8 < 17). 17=16+1? si, quindi abbiamo trovato to be. A questo punto passo al secondo merge con or facendo la stessa cosa rispetto a be, e così via. Se poi non c'era to be or not to be rispetto alle posizioni 16, 17, 18, 19, 20, allora ripartiamo dal merge tra to e be per cercare un altro to be (es. quello in posizione 190 e 191) e così via fino a trovare la frase esatta o esaurire le posting list.

I positional index inoltre permettono di gestire query più complesse oltre alle phrase queries booleane, come le **proximity queries**. Queste sono queries che chiedono che una certa parola sia in prossimità k rispetto ad un'altra parola. Ad esempio STATUTE /3 FEDERAL significa che la parola STATUTE deve apparire entro 3 parole da FEDERAL. Per gestire questo tipo di query, si fa un merge simile a quello delle phrase queries, ma invece di cercare posizioni consecutive (*pos2 = pos1 + 1*), si cerca che la differenza tra le posizioni sia minore o uguale a k (*|pos2 - pos1| <= k*).

es. statute : 10, 40, 80 federal : 12, 70, statute /3 federal, confrontiamo prima 10 vs 12: |12 - 10| <= 3? si, quindi abbiamo trovato un match. Poi scorriamo statute (perché 10 < 12) e confrontiamo 40 vs 12: |12 - 40| <= 3? no, quindi scorriamo federal etc..

Si noti che modificando il merge come descritto, il costo resta lineare nelle due liste di posizioni.

Ora ci chiediamo, **ma quanto ci costa aggiungere tutti questi indici di posizioni**? Parecchio. Dovendo salvare per ogni parola tutte le posizioni, il numero totale di posizioni salvate è pari al numero totale di parole in tutti i documenti, che è molto più grande del numero di termini distinti. Tuttavia salvare una parola richiede più byte rispetto a salvare una posizione, dal momento che una posizione può essere rappresentata con un numero intero mentre una parola richiede in media 6 byte. 

La dimensione del positional index dipende molto dalla dimensione dei documenti: immaginiamo che una parola rara appare con frequenza 0.1% (una volta ogni 1000 parole). Allora, in un documento di 1000 parole, quella parola apparirà in media una volta, quindi dovremo salvare una posizione per quel documento. In un documento di 100.000 parole, quella parola apparirà in media 100 volte, quindi dovremo salvare 100 posizioni per quel documento. Quindi più i documenti sono lunghi, più il positional index diventa grande.

Rules of thumb: 
- in generale, un positional index è circa 2-4 volte pi+ grande di un inverted index classico (term -> list of docs)
- la dimensione dell'indice posizionale è circa 35-50% della dimensione dei documenti originali (in termini di byte), quindi è un indice molto grande, ma che permette di gestire query molto più complesse e precise rispetto all'inverted index classico. es. se il corpus originale è di 100 GB, solo l'indice posizionale potrebbe essere di 35-50 GB.

Nonostante il costo significativo, oggi quasi tutti i motori di ricerca usano indici posizionali, in quanto permettono di gestire query molto più complesse e precise, migliorando significativamente la qualità dei risultati restituiti agli utenti.

**Schema Ibrido**: alcune frasi sono molto comuni, per esempio "Michael Jackson", "New York", "The Who". Se usassimo il positional index per cercarle, dovremmo ogni volta fare il merge delle posizioni, e questo può essere molto lento. Per questo motivo alcuni sistemi fanno un indice misto: si usa il positional index normale e in più si mantengono quelle frasi comuni come termini distinti nell'inverted index classico. In questo modo quando si cerca la frase in questione può essere restituita direttamente la posting list ad essa associata senza fare operazioni di merge sulle posizioni.

Uno studio mostra come usando questo approccio, nonostante uno spazio aggiuntivo del 26% per mantenere le frasi comuni, il tempo di risposta alle query è in media 1/4 rispetto a un sistema che usa solo il positional index. Si parla quindi di tradeoff tra spazio e tempo di risposta.